In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls /content/drive/MyDrive


In [ ]:
!find /content/drive/MyDrive -name "*.zip"


In [ ]:
!unzip "/content/drive/MyDrive/archive (2).zip" -d /content/udacity_data


In [ ]:
!ls /content/udacity_data


In [ ]:
!ls /content/udacity_data/data/export


In [ ]:
IMAGE_DIR = "/content/udacity_data/data/export"


In [ ]:
import os
import random
import shutil

base_dir = "/content/udacity_data/data"
src_dir = os.path.join(base_dir, "export")

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

images = os.listdir(src_dir)
random.shuffle(images)

split = int(0.8 * len(images))
train_imgs = images[:split]
val_imgs = images[split:]

for img in train_imgs:
    shutil.copy(os.path.join(src_dir, img), os.path.join(train_dir, img))

for img in val_imgs:
    shutil.copy(os.path.join(src_dir, img), os.path.join(val_dir, img))

print("Train images:", len(train_imgs))
print("Val images:", len(val_imgs))


In [ ]:
!pip uninstall -y transformers
!pip install -q transformers datasets pillow torch torchvision

In [ ]:
import os
import random

TRAIN_DIR = "/content/udacity_data/data/train"
VAL_DIR = "/content/udacity_data/data/val"

captions = [
    "A road scene with vehicles",
    "Urban street with lane markings",
    "Driving scene with trees and buildings",
    "Autonomous driving road environment",
    "Street view captured from a vehicle"
]

def create_dataset(image_dir):
    data = []
    for img in os.listdir(image_dir):
        data.append({
            "image": os.path.join(image_dir, img),
            "text": random.choice(captions)
        })
    return data

train_data = create_dataset(TRAIN_DIR)
val_data = create_dataset(VAL_DIR)

print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset


In [ ]:
from transformers import ViltProcessor
import torch
from PIL import Image

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

def preprocess(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image"]]

    encoding = processor(
        images=images,
        text=batch["text"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

    encoding["labels"] = torch.zeros(len(images), dtype=torch.long)
    return encoding


In [ ]:
import os
import random

TRAIN_DIR = "/content/udacity_data/data/export"
VAL_DIR = "/content/udacity_data/data/export"  # using same folder for demo

captions = [
    "A road scene with vehicles",
    "Urban street with lane markings",
    "Driving scene with trees and buildings",
    "Autonomous driving road environment",
    "Street view captured from a vehicle"
]

def create_dataset(image_dir):
    data = []
    for img in os.listdir(image_dir):
        if img.endswith(".jpg"):
            data.append({
                "image": os.path.join(image_dir, img),
                "text": random.choice(captions)
            })
    return data

train_data = create_dataset(TRAIN_DIR)
val_data = create_dataset(VAL_DIR)

print("Train samples:", len(train_data))
print("Val samples:", len(val_data))


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset


In [ ]:
train_dataset = train_dataset.shuffle(seed=42).select(range(100))
val_dataset = val_dataset.shuffle(seed=42).select(range(40))


In [ ]:
from transformers import ViltProcessor
from PIL import Image
import torch

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")

def preprocess(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image"]]
    encoding = processor(
        images=images,
        text=batch["text"],
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    # For classification, we provide 'labels'. Using dummy labels for now.
    encoding["labels"] = torch.zeros(len(images), dtype=torch.long) # Assuming a binary classification, all to class 0 for now.
    return encoding

In [ ]:
train_dataset = train_dataset.map(preprocess, batched=True, batch_size=2)
val_dataset = val_dataset.map(preprocess, batched=True, batch_size=2)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")


In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./vilt_results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    logging_steps=10,
    num_train_epochs=1,
    save_strategy="no",
    report_to="none",
    label_names=["labels"] # Changed to 'labels' for classification task
)

In [ ]:
!pip install -U transformers accelerate


In [ ]:
from transformers import ViltForQuestionAnswering, ViltProcessor

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-finetuned-vqa")
model = ViltForQuestionAnswering.from_pretrained(
    "dandelin/vilt-b32-finetuned-vqa"
)


In [ ]:
def make_qa(text):
    return {
        "question": "What is shown in the driving scene?",
        "answers": ["road", "street", "vehicles"]
    }

train_dataset = train_dataset.map(
    lambda x: {
        "question": "What is shown in the driving scene?",
        "answers": {"text": ["road"], "answer_start": [0]}
    }
)

val_dataset = val_dataset.map(
    lambda x: {
        "question": "What is shown in the driving scene?",
        "answers": {"text": ["road"], "answer_start": [0]}
    }
)


In [ ]:
import torch
from PIL import Image

def preprocess_vqa(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image"]]

    enc = processor(
        images=images,
        text=batch["question"],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # ViltForQuestionAnswering expects multi-hot encoded labels matching the size of its answer head.
    # The size is model.config.num_labels (which is 3129 for this pre-trained model).
    num_labels = model.config.num_labels

    # Initialize labels as zeros with the correct shape and dtype (float for BCEWithLogitsLoss)
    labels = torch.zeros(len(images), num_labels, dtype=torch.float)

    # For actual VQA training, you would parse batch["answers"] and map them to the model's
    # answer vocabulary, then set the corresponding indices in 'labels' to 1.0.
    # For this example, we are just ensuring the correct shape to resolve the ValueError.
    # If you have specific answers and a mapping, you would implement that logic here.
    # For now, we will assign a dummy value to the first possible answer for demonstration.
    # For instance, if 'road' corresponds to index 0 in the model's vocabulary:
    # labels[:, 0] = 1.0

    enc["labels"] = labels
    return enc

In [ ]:
train_dataset = train_dataset.map(preprocess_vqa, batched=True, batch_size=2)
val_dataset = val_dataset.map(preprocess_vqa, batched=True, batch_size=2)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")


In [ ]:
import torch
from PIL import Image

NUM_ANSWERS = model.config.num_labels  # 3129

def preprocess_vqa(batch):
    images = [Image.open(p).convert("RGB") for p in batch["image"]]

    enc = processor(
        images=images,
        text=batch["question"],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # Create dummy multi-hot labels (correct shape)
    labels = torch.zeros((len(images), NUM_ANSWERS))
    labels[:, 0] = 1.0   # mark first answer as true (dummy)

    enc["labels"] = labels
    return enc


In [ ]:
train_dataset = train_dataset.map(preprocess_vqa, batched=True, batch_size=2)
val_dataset = val_dataset.map(preprocess_vqa, batched=True, batch_size=2)

train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")


In [ ]:
from transformers import ViltProcessor, ViltModel
import torch
from PIL import Image

processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")
model = ViltModel.from_pretrained("dandelin/vilt-b32-mlm")
model.eval()


In [ ]:
captions = [
    "A road scene with vehicles",
    "An empty highway",
    "A city street with traffic",
    "A rural road",
    "A non-driving indoor scene"
]


In [ ]:
import numpy as np

def compute_similarity(image_path, texts):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(
        images=image,
        text=texts,
        return_tensors="pt",
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    image_emb = outputs.pooler_output[0]
    text_embs = outputs.pooler_output[1:]

    scores = torch.matmul(text_embs, image_emb)
    return scores.numpy()


In [ ]:
import torch
from PIL import Image
import numpy as np

def compute_similarity(image_path, texts):
    image = Image.open(image_path).convert("RGB")

    # Repeat the image for each text
    images = [image] * len(texts)

    inputs = processor(
        images=images,
        text=texts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Use pooled output as embedding
    embeddings = outputs.pooler_output  # shape: (batch_size, hidden_dim)

    image_emb = embeddings[0]
    text_embs = embeddings[1:]

    # Cosine similarity
    scores = torch.nn.functional.cosine_similarity(
        text_embs, image_emb.unsqueeze(0), dim=1
    )

    return scores.numpy()


In [ ]:
import random
import numpy as np

sample_images = random.sample(train_data, 10)

results = []

for sample in sample_images:
    scores = compute_similarity(sample["image"], captions)
    predicted = captions[np.argmax(scores)]

    results.append({
        "image": sample["image"],
        "predicted_caption": predicted,
        "scores": scores
    })

results


In [ ]:
avg_score = np.mean([max(r["scores"]) for r in results])
avg_score


In [ ]:
image_path = "/content/udacity_data/data/export/1478900057319186387_jpg.rf.2b0a0b492115d38887004002f572dd8d.jpg"  # Corrected path with full filename

scores = compute_similarity(image_path, captions)
predicted_caption = captions[np.argmax(scores)]

result = {
    "image": image_path,
    "predicted_caption": predicted_caption,
    "scores": scores
}

result

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

# Load image
img = Image.open(result["image"]).convert("RGB")

# Display
plt.figure(figsize=(6,4))
plt.imshow(img)
plt.axis("off")
plt.title(f"Predicted Caption: {result['predicted_caption']}", fontsize=12)
plt.show()


In [ ]:
results = [
    {
        "image": "/content/udacity_data/data/export/1478900057319186387_jpg.rf.2b0a0b492115d38887004002f572dd8d.jpg",
        "predicted_caption": "An empty highway"
    },
    {
        "image": "/content/udacity_data/data/export/1478895399594026807_jpg.rf.359466f82cb68195c1fa5cd24c3bd4ad.jpg",
        "predicted_caption": "A city street with traffic"
    }
]

for r in results:
    img = Image.open(r["image"]).convert("RGB")
    plt.figure(figsize=(6,4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Predicted Caption: {r['predicted_caption']}", fontsize=12)
    plt.show()
